# Local Differential Privacy (LDP) for Vector Search

A local port of the Colab notebook demonstrating how **Local Differential Privacy** can be
applied to query embeddings before they reach a vector search engine (e.g. the OpenSearch k-NN plugin).

The idea: instead of sending the user's raw query embedding to the search backend, where it
can be inverted to recover the original intent. Instead the client injects calibrated **Laplace noise**
into the vector first. The engine still returns useful results, but no single query reveals
exactly what the user was looking for.

## Running locally

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
jupyter lab   # then open this notebook
```

The first run downloads the `all-MiniLM-L6-v2` model (~90 MB) from Hugging Face and caches it
under `~/.cache/huggingface`. Everything after that runs offline.

## 1. Load the embedding model

In [1]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model once for global use.
# Downloaded and cached on first run (~90 MB).
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Set up the simulated search index

These 20 product names stand in for documents already indexed in OpenSearch. We embed them,
L2-normalize them, then reduce them to 20 dimensions with PCA. The same PCA model is later
applied to the query, so both live in the same space.

In [2]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Fixed seed so the noisy results in this notebook are reproducible.
np.random.seed(42)

# Setup Simulation Data (The "Search Index")
# Imagine these are document embeddings already in OpenSearch
documents = [
    "Laptop",
    "Smartphone",
    "Tablet",
    "Headphones",
    "Monitor",
    "Smartwatch",
    "Wireless Earbuds",
    "Gaming Console",
    "E-Reader",
    "Projector",
    "External Hard Drive",
    "Webcam",
    "Keyboard",
    "Mouse",
    "Printer",
    "Router",
    "Speaker",
    "Microphone",
    "Drone",
    "VR Headset",
]
doc_names = documents

# Generate high-dimensional embeddings for documents
doc_vectors_full = embedding_model.encode(doc_names)
doc_vectors_full = normalize(doc_vectors_full)

# Reduce dimensionality so the noise budget is spread over fewer coordinates
pca_model = PCA(n_components=20)
doc_vectors = pca_model.fit_transform(doc_vectors_full)

print(f"Original document vector dimension: {doc_vectors_full.shape[1]}")
print(f"PCA-reduced document vector dimension: {doc_vectors.shape[1]}")

Original document vector dimension: 384
PCA-reduced document vector dimension: 20


## 3. The LDP engine

This is the logic that would live client-side (e.g. in `ubi.js`). Lower `epsilon` means a
larger noise scale, which means more privacy and less search accuracy.

In [4]:
def inject_laplace_noise(vector, epsilon, sensitivity=1.0):
    """
    Adds Laplace noise to each coordinate of a vector.
    Lower epsilon = Higher Privacy = More Noise.
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale, size=vector.shape)
    return vector + noise

## 4. Generate query embeddings

Simulating what a real-world embedding model would produce for a user's typed query.

In [5]:
# Example text queries
text_queries = [
    "laptop computer",
]

# Try these instead to see how different intents behave:
# text_queries = [
#     "laptop for coding",
#     "best smartphone for photography",
#     "comfortable headphones",
#     "gaming monitor review",
# ]

query_embeddings = embedding_model.encode(text_queries)
query_embeddings = normalize(query_embeddings)

print("Generated Embeddings for Queries:")
for i, query in enumerate(text_queries):
    print(f"Query: '{query}'\nEmbedding (first 5 dims): {query_embeddings[i][:5]}...")
    print(f"Embedding Dimension: {len(query_embeddings[i])}")

Generated Embeddings for Queries:
Query: 'laptop computer'
Embedding (first 5 dims): [-0.06790411  0.04620177  0.03757743 -0.04531892  0.0187656 ]...
Embedding Dimension: 384


## 5. Privatize the query vector

Project the raw query through the *fitted* PCA model, then add Laplace noise. Only the noised
vector ever leaves the client.

In [6]:
# Pick the embedding for 'laptop computer'
raw_query_vector_from_model_full = query_embeddings[0]

epsilon = 1.2  # the privacy control

# Apply the *fitted* PCA model to the raw query vector
raw_query_vector_from_model = pca_model.transform(
    raw_query_vector_from_model_full.reshape(1, -1)
)[0]

# Generate the 'Privacy-Preserving' query on the PCA-reduced vector
noised_query_vector_from_model = inject_laplace_noise(raw_query_vector_from_model, epsilon)

print(f"Original Query ('laptop computer' embedding - full dims): {raw_query_vector_from_model_full[:5]}...")
print(f"PCA-reduced Query (first 5 dims): {raw_query_vector_from_model[:5]}...")
print(f"Noised Query (first 5 dims):      {noised_query_vector_from_model[:5]}...")
print(f"Dimension of noised query vector: {len(noised_query_vector_from_model)}")

Original Query ('laptop computer' embedding - full dims): [-0.06790411  0.04620177  0.03757743 -0.04531892  0.0187656 ]...
PCA-reduced Query (first 5 dims): [ 0.23512647 -0.10999608  0.05589662 -0.1570354   0.01126605]...
Noised Query (first 5 dims):      [-0.00563118  1.82081579  0.57556205  0.02612742 -0.95926113]...
Dimension of noised query vector: 20


/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: divide by zero encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: overflow encountered in 

## 6. k-NN search

Simulating the OpenSearch k-NN plugin. Both the documents and the query live in the same
PCA-reduced space, so the noised query can be searched directly.

In [7]:
k_nearest_neighbors = NearestNeighbors(n_neighbors=5, metric='euclidean')
k_nearest_neighbors.fit(doc_vectors)
distances_knn, indices_knn = k_nearest_neighbors.kneighbors([noised_query_vector_from_model])

print("--- Search Results (using noised query vector) ---")
for i, idx in enumerate(indices_knn[0]):
    print(f"Result {i+1}: {doc_names[idx]} (Distance: {distances_knn[0][i]:.4f})")

--- Search Results (using noised query vector) ---
Result 1: Gaming Console (Distance: 4.8870)
Result 2: Webcam (Distance: 4.9303)
Result 3: Mouse (Distance: 4.9658)
Result 4: Smartwatch (Distance: 5.0329)
Result 5: Drone (Distance: 5.0358)


In a traditional engine, we aim for Precision@1. In a Privacy-Preserving Engine, we optimize
for Recall@5. We accept that the 'True Intent' might be at position #2 or #3, knowing that the
aggregate results still provide the user what they need while the 'noise' provides the legal
and ethical cover they require.

## 7. The privacy/utility tradeoff: sweeping epsilon

A single epsilon only shows one point on the curve. Here we sweep it and measure where the
**true intent** (`"Laptop"`) actually ranks in the results, averaged over many noisy draws of
the same query.

Two metrics, the ones the framing above contrasts:

- **P@1**: how often the true intent is the top result (what a traditional engine optimizes).
- **R@5**: how often it appears anywhere in the top 5 (what a privacy-preserving engine
  settles for).

Low epsilon destroys both. High epsilon recovers both, and gives the attacker their answer
back. The interesting region is in between, where R@5 is high but P@1 has collapsed: the user
still finds what they need, but no single query pins down the intent.

In [9]:
TRUE_INTENT = "Laptop"
true_idx = doc_names.index(TRUE_INTENT)

epsilons = [0.5, 1.2, 3, 5, 10, 20, 50, 100]
n_trials = 200

# Rank against the whole index so we can find the true intent wherever it lands
sweep_knn = NearestNeighbors(n_neighbors=len(doc_names), metric='euclidean')
sweep_knn.fit(doc_vectors)

np.random.seed(0)  # reproducible sweep

sweep = []
for eps in epsilons:
    ranks = []
    for _ in range(n_trials):
        noised = inject_laplace_noise(raw_query_vector_from_model, eps)
        _, idx = sweep_knn.kneighbors([noised])
        ranks.append(int(np.where(idx[0] == true_idx)[0][0]) + 1)
    ranks = np.array(ranks)
    sweep.append({
        'epsilon': eps,
        'mean_rank': ranks.mean(),
        'p_at_1': (ranks == 1).mean(),
        'r_at_5': (ranks <= 5).mean(),
    })

print(f"Rank of '{TRUE_INTENT}' over {n_trials} noisy queries per epsilon "
      f"(index size: {len(doc_names)})")
print(f"{'epsilon':>8} {'mean rank':>10} {'P@1':>7} {'R@5':>7}")
for row in sweep:
    print(f"{row['epsilon']:>8} {row['mean_rank']:>10.2f} "
          f"{row['p_at_1']:>7.2f} {row['r_at_5']:>7.2f}")

print(f"\nRandom-guess baseline: mean rank {(len(doc_names) + 1) / 2:.1f}, "
      f"P@1 {1 / len(doc_names):.2f}, R@5 {5 / len(doc_names):.2f}")

Rank of 'Laptop' over 200 noisy queries per epsilon (index size: 20)
 epsilon  mean rank     P@1     R@5
     0.5       9.15    0.04    0.27
     1.2       7.05    0.07    0.45
       3       3.94    0.38    0.75
       5       1.84    0.72    0.95
      10       1.03    0.98    1.00
      20       1.00    1.00    1.00
      50       1.00    1.00    1.00
     100       1.00    1.00    1.00

Random-guess baseline: mean rank 10.5, P@1 0.05, R@5 0.25


In [ ]:
# Plot the tradeoff curve
eps_vals = [r['epsilon'] for r in sweep]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eps_vals, [r['p_at_1'] for r in sweep], 'o-',
        label='P@1 (the correct item is the top hit)')
ax.plot(eps_vals, [r['r_at_5'] for r in sweep], 's-',
        label='R@5 (the correct item is in the top five)')
ax.axhline(5 / len(doc_names), color='gray', linestyle=':',
           label='R@5 if you guessed at random')
ax.axvline(epsilon, color='red', linestyle='dashed', linewidth=2,
           label=f'epsilon {epsilon}, used in the demo above')
ax.set_xscale('log')
# A log axis labels itself 10^0, 10^1, 10^2, which leaves the demo's epsilon 1.2
# — the red line, and the whole point of this chart — sitting between unlabelled
# ticks. Label the sampled values instead.
ax.set_xticks(eps_vals)
ax.set_xticklabels([f'{t:g}' for t in eps_vals])
ax.minorticks_off()
ax.set_xlabel('Epsilon (higher = less noise, less privacy)')
ax.set_ylabel(f"Fraction of noisy queries that find '{TRUE_INTENT}'")
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"Does the noised query still find '{TRUE_INTENT}'?")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig('plots/07_epsilon_tradeoff_toy_index.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. Visualization: the "statistical tent" audit

Draw 1,000 independent noisy versions of the same query and histogram the first coordinate.
The result should be the characteristic Laplace "tent" centered on the true value. This is
the auditable proof that the noise mechanism behaves as claimed.

In [ ]:
def plot_audit():
    # Generate 1000 noisy samples to verify the Laplace distribution
    samples = [
        inject_laplace_noise(raw_query_vector_from_model, epsilon)[0]
        for _ in range(1000)
    ]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(samples, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    ax.axvline(
        raw_query_vector_from_model[0],
        color='red', linestyle='dashed', linewidth=2,
        label='the true value, before noise',
    )
    ax.set_title(
        f"One coordinate of one query, privatized 1,000 times (epsilon = {epsilon})"
    )
    ax.set_xlabel('Value of that coordinate after noise')
    ax.set_ylabel('Number of draws, out of 1,000')
    ax.legend()
    fig.savefig('plots/08_laplace_tent_audit.png', dpi=200, bbox_inches='tight')
    plt.show()


plot_audit()


In [ ]:
# The same audit at two epsilons. The x axis is shared on purpose: let each panel
# autoscale and both look like the same tent, which is exactly the wrong lesson.
COMPARE_EPSILON = 10

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
draws = {
    eps: [inject_laplace_noise(raw_query_vector_from_model, eps)[0] for _ in range(1000)]
    for eps in (epsilon, COMPARE_EPSILON)
}
lo, hi = min(draws[epsilon]), max(draws[epsilon])
edges = np.linspace(lo, hi, 51)

for ax, eps in zip(axes, (epsilon, COMPARE_EPSILON)):
    ax.hist(draws[eps], bins=edges, color='skyblue', edgecolor='black', alpha=0.7)
    ax.axvline(
        raw_query_vector_from_model[0],
        color='red', linestyle='dashed', linewidth=2,
        label='the true value, before noise',
    )
    ax.set_title(f"epsilon = {eps}   (noise scale {1 / eps:.3f})")
    ax.set_xlabel('Value of that coordinate after noise')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Number of draws, out of 1,000')
axes[0].legend(loc='upper left')
fig.suptitle('Same coordinate, same 1,000 draws, two privacy settings')
fig.tight_layout()
fig.savefig('plots/08b_epsilon_spread_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. Privacy failure vs. privacy success

This section demonstrates the effectiveness of LDP by comparing the results of a simulated
'vector inversion' attack on both a raw query vector and a differentially private (LDP-noised)
query vector. A 'vector inversion' tool here is simulated by performing a k-NN search on the
given vector against our document index, aiming to infer the original intent.

In [8]:
# Simulate 'Privacy Failure': Vector Inversion on Raw Query
k_nearest_neighbors_raw = NearestNeighbors(n_neighbors=2, metric='euclidean')
k_nearest_neighbors_raw.fit(doc_vectors)
distances_raw, indices_raw = k_nearest_neighbors_raw.kneighbors([raw_query_vector_from_model])

print("--- Privacy Failure: Inferred from RAW Query Vector ---")
print("Attacker's 'guess' of user's intent (high accuracy):")
for i, idx in enumerate(indices_raw[0]):
    print(f"Result {i+1}: {doc_names[idx]} (Distance: {distances_raw[0][i]:.4f})")

# Simulate 'Privacy Success': Vector Inversion on LDP-Noised Query
k_nearest_neighbors_noised = NearestNeighbors(n_neighbors=2, metric='euclidean')
k_nearest_neighbors_noised.fit(doc_vectors)
distances_noised, indices_noised = k_nearest_neighbors_noised.kneighbors(
    [noised_query_vector_from_model]
)

print("\n--- Privacy Success: Inferred from LDP-Noised Query Vector ---")
print("Attacker's 'guess' of user's intent (low accuracy/gibberish):")
for i, idx in enumerate(indices_noised[0]):
    print(f"Result {i+1}: {doc_names[idx]} (Distance: {distances_noised[0][i]:.4f})")

--- Privacy Failure: Inferred from RAW Query Vector ---
Attacker's 'guess' of user's intent (high accuracy):
Result 1: Laptop (Distance: 0.2092)
Result 2: Keyboard (Distance: 0.9104)

--- Privacy Success: Inferred from LDP-Noised Query Vector ---
Attacker's 'guess' of user's intent (low accuracy/gibberish):
Result 1: Gaming Console (Distance: 4.8870)
Result 2: Webcam (Distance: 4.9303)


---

## 10. Scaling up: a real 43,000-product index

Everything above ran against 20 unrelated product names. That index is small enough to hold in
your head, which makes the mechanism easy to see, but it distorts the privacy result. With 20
unrelated items, noise pushes the query to a *random* item, so "the attacker learned nothing"
is partly just an artifact of the index having no near-neighbours.

Here we swap in **WANDS**, Wayfair's product search relevance dataset (ECIR 2022, MIT
licensed): 42,994 real product names across 861 product classes. Nothing about the mechanism
changes. Only the index does.

The payoff is that WANDS ships a `product_class` label per product, which lets us separate two
very different questions an attacker might be asking:

- **Item-level recovery**: did they recover the *exact product* the user searched for?
- **Class-level recovery**: did they recover the *category* (Beds, Area Rugs, Accent Chairs)?

That distinction is invisible in a 20-item toy index and is the one that actually matters.

> **Run `python download_wands.py` once before this section.** It downloads the dataset,
> embeds all 43k products, and caches the vectors so this notebook needs no network.

### 10a. Fetch and cache the dataset

This runs `download_wands.py` for you. It downloads the WANDS files, embeds all 43k product
names, fits the PCA basis, and writes everything to `data/` as plain numpy files.

It uses the same interpreter as this kernel and needs no libraries beyond the ones sections 1
through 9 already use, so if those sections ran, this will too.

The script is idempotent. Anything already cached is skipped, so re-running this cell between
rehearsals costs nothing. Expect roughly 30 to 60 seconds on the first run, almost all of it
the one-time download.

**Run this cell before presenting, not during.** After it has been run once, the rest of the
notebook needs no network at all.

In [13]:
import subprocess
import sys

# The script runs in the SAME interpreter as this kernel, so whatever works here works there
print(f"Kernel interpreter: {sys.executable}\n")

process = subprocess.Popen(
    [sys.executable, 'download_wands.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line.rstrip())

if process.wait() != 0:
    raise RuntimeError(
        "download_wands.py failed. Check the output above. The usual causes are no network "
        "access, or a kernel whose interpreter is missing numpy, scikit-learn or "
        "sentence-transformers. The path printed above is the interpreter it used."
    )

Kernel interpreter: /work/jzonthemtn/ldp-for-search/venv/bin/python3

product.csv: already cached
query.csv: already cached
embeddings: already cached

Done. The notebook can now run fully offline.


In [14]:
from pathlib import Path

DATA_DIR = Path('data')

if not (DATA_DIR / 'wands_vectors_pca20.npy').exists():
    raise FileNotFoundError("WANDS cache not found. Run the cell above first.")

# Pre-computed by download_wands.py: same model, same PCA dimensionality as above.
# Stored as plain .npy/.npz so section 10 needs no libraries beyond the ones already in use.
wands_vectors = np.load(DATA_DIR / 'wands_vectors_pca20.npy')
wands_pca = np.load(DATA_DIR / 'wands_pca.npz')

_products = np.load(DATA_DIR / 'wands_products.npz')
wands_names = _products['names']
wands_classes = _products['classes']

print(f"Index size:      {len(wands_vectors):,} products")
print(f"Product classes: {len(np.unique(wands_classes)):,}")
print(f"Vector shape:    {wands_vectors.shape}")
print("\nSample products:")
for name in wands_names[:5]:
    print(f"  - {name}")

Index size:      42,994 products
Product classes: 861
Vector shape:    (42994, 20)

Sample products:
  - solid wood platform bed
  - all-clad 7 qt . slow cooker
  - all-clad electrics 6.5 qt . slow cooker
  - all-clad all professional tools pizza cutter
  - baldwin prestige alcott passage knob with round rosette


### 10b. The attack at scale

First, the baseline: what does an attacker recover from the **raw** query vector against a real
index? This is the same 'vector inversion' simulation as section 9, just with 43k candidates
instead of 20.

In [15]:
def wands_project(text):
    """Embed a query and project it into the cached WANDS PCA space."""
    full = normalize(embedding_model.encode([text]))[0]
    return (full - wands_pca['mean']) @ wands_pca['components'].T


def wands_topk(vector, k=5):
    """Return the indices of the k nearest products, nearest first."""
    distances = np.linalg.norm(wands_vectors - vector, axis=1)
    top = np.argpartition(distances, k)[:k]
    return top[np.argsort(distances[top])], distances


WANDS_QUERY = "solid wood platform bed"

wands_raw_query = wands_project(WANDS_QUERY)
raw_top, raw_distances = wands_topk(wands_raw_query, k=5)

# The attacker's target: what the RAW query resolves to
true_item_name = wands_names[raw_top[0]]
true_item_class = wands_classes[raw_top[0]]

print(f"Query: '{WANDS_QUERY}'")
print("\n--- Privacy Failure: attacker inverts the RAW query vector ---")
for rank, idx in enumerate(raw_top, start=1):
    print(f"  {rank}. {wands_names[idx][:52]:<52} "
          f"[{wands_classes[idx][:22]}]  d={raw_distances[idx]:.3f}")

print(f"\nTrue intent -> item:  '{true_item_name}'")
print(f"            -> class: '{true_item_class}'")

Query: 'solid wood platform bed'

--- Privacy Failure: attacker inverts the RAW query vector ---
  1. solid wood platform bed                              [Beds]  d=0.000
  2. solid wood platform bed                              [Beds]  d=0.000
  3. devery solid wood platform bed                       [Beds]  d=0.099
  4. wynd solid wood platform bed                         [Beds]  d=0.146
  5. travie solid wood platform bed                       [Beds]  d=0.149

True intent -> item:  'solid wood platform bed'
            -> class: 'Beds'


### 10c. Item-level vs class-level recovery

Now sweep epsilon again, but score the attacker on both questions. A match counts at the
**item** level when the noised query's top hit has the same product name as the raw query's top
hit (WANDS contains genuine duplicate listings, so comparing names rather than row positions
avoids penalising the attacker for landing on an identical product), and at the **class** level
when it merely lands in the same category.

Note the `scale` column: Laplace scale is `sensitivity / epsilon`, and what matters is its size
*relative to the spread of the vector coordinates*. That spread differs between the 20-document
PCA and this 43k-document one, which is why the useful epsilon range here is nothing like
`1.2`. Epsilon is not portable across indexes. That is an important caveat when quoting a privacy
budget.

In [16]:
wands_epsilons = [1, 5, 10, 20, 30, 50, 75, 100, 200]
n_trials = 300

coord_std = wands_vectors.std(axis=0).mean()
print(f"Mean per-coordinate spread of the index: {coord_std:.3f}")
print(f"Query: '{WANDS_QUERY}'  |  {n_trials} noisy draws per epsilon\n")

header = f"{'epsilon':>8}{'scale':>8}{'item P@1':>10}{'item R@5':>10}{'class P@1':>11}{'class R@5':>11}"
print(header)
print('-' * len(header))

wands_sweep = []
for eps in wands_epsilons:
    rng = np.random.default_rng(0)  # same draws at every epsilon, so rows are comparable
    noised = wands_raw_query + rng.laplace(0, 1.0 / eps, size=(n_trials, wands_vectors.shape[1]))

    # Distance from every noisy query to every product, then the top 5 of each row
    dists = np.linalg.norm(wands_vectors[None, :, :] - noised[:, None, :], axis=2)
    top5 = np.argpartition(dists, 5, axis=1)[:, :5]
    top5 = np.take_along_axis(top5, np.argsort(np.take_along_axis(dists, top5, 1), axis=1), axis=1)
    top1 = top5[:, 0]

    row = {
        'epsilon': eps,
        'scale': 1.0 / eps,
        'item_p1': (wands_names[top1] == true_item_name).mean(),
        'item_r5': (wands_names[top5] == true_item_name).any(axis=1).mean(),
        'class_p1': (wands_classes[top1] == true_item_class).mean(),
        'class_r5': (wands_classes[top5] == true_item_class).any(axis=1).mean(),
    }
    wands_sweep.append(row)
    print(f"{eps:>8}{row['scale']:>8.3f}{row['item_p1']:>10.2f}{row['item_r5']:>10.2f}"
          f"{row['class_p1']:>11.2f}{row['class_r5']:>11.2f}")

n_docs = len(wands_vectors)
class_size = int((wands_classes == true_item_class).sum())
print(f"\nRandom-guess baseline over {n_docs:,} products:")
print(f"  item P@1  {1 / n_docs:.5f}   class P@1  {class_size / n_docs:.5f} "
      f"('{true_item_class}' has {class_size:,} members)")

Mean per-coordinate spread of the index: 0.128
Query: 'solid wood platform bed'  |  300 noisy draws per epsilon

 epsilon   scale  item P@1  item R@5  class P@1  class R@5
----------------------------------------------------------
       1   1.000      0.00      0.01       0.06       0.11
       5   0.200      0.04      0.09       0.33       0.60
      10   0.100      0.13      0.31       0.63       0.92
      20   0.050      0.42      0.76       0.82       1.00
      30   0.033      0.70      0.96       0.93       1.00
      50   0.020      0.93      1.00       0.98       1.00
      75   0.013      0.99      1.00       1.00       1.00
     100   0.010      1.00      1.00       1.00       1.00
     200   0.005      1.00      1.00       1.00       1.00

Random-guess baseline over 42,994 products:
  item P@1  0.00002   class P@1  0.02586 ('Beds' has 1,112 members)


In [ ]:
# The gap between the two curves IS the privacy story
eps_vals = [r['epsilon'] for r in wands_sweep]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(eps_vals, [r['class_p1'] for r in wands_sweep], 's-', color='tab:orange',
        label='the category the user searched in (Beds, Area Rugs, ...)')
ax.plot(eps_vals, [r['item_p1'] for r in wands_sweep], 'o-', color='tab:blue',
        label='the exact product the user searched for')

ax.fill_between(
    eps_vals,
    [r['item_p1'] for r in wands_sweep],
    [r['class_p1'] for r in wands_sweep],
    color='tab:orange', alpha=0.15,
    label='the gap, where the category is known but the item is not',
)

ax.set_xscale('log')
# A log axis labels itself 10^0, 10^1, 10^2, and the slide captions cite specific
# epsilons. Label the sampled values instead, so "at epsilon 1" is readable off
# the axis rather than converted in the audience's head.
eps_ticks = [1, 5, 10, 20, 50, 100, 200]
ax.set_xticks(eps_ticks)
ax.set_xticklabels([str(t) for t in eps_ticks])
ax.minorticks_off()
ax.set_xlabel('Epsilon (higher = less noise, less privacy)')
ax.set_ylabel('Fraction of queries the attacker recovers')
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"What an attacker recovers from a noised query ({len(wands_vectors):,} products)")
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
fig.savefig('plots/10_item_vs_class_recovery.png', dpi=200, bbox_inches='tight')
plt.show()


### What the larger index shows that the toy index could not

The two curves separate. Around **epsilon = 10 to 20** the attacker recovers the *category* most
of the time while the *exact product* stays largely hidden. They can tell this person was
shopping for beds, but not which bed, out of a thousand-plus candidates.

That gap is the honest version of the privacy claim, and it is a much more defensible one than
"the attacker gets gibberish." It also states the residual leak plainly: LDP at a usable
epsilon does **not** hide the broad category of what someone is shopping for. If category
alone is sensitive, say a medical or legal corpus rather than furniture, then this mechanism
at this epsilon is not sufficient, and that is worth saying out loud rather than leaving for someone
to find in the Q&A.

---

## 11. What survives: aggregate trends

Every result so far measures what LDP *costs*. Section 7 showed P@1 collapsing. Section 10
showed the attacker losing the exact item. Taken alone, that is a story of pure loss, and it
would be a strange argument for adopting the technique.

The reason LDP is useful is the asymmetry it creates. The noise is zero-mean, so it cancels
when you average over many independent users. One person's query is unrecoverable. The trend
across ten thousand people is not.

That asymmetry is the whole basis for keeping relevance work alive under a no-raw-queries
policy. Learning-to-rank training data, query-intent clustering and demand trends are all
aggregate statistics. None of them need any individual query to be readable.

This section makes that concrete. We build cohorts of users from the **480 real WANDS queries**,
have every user privatize their own query on their own device, and then ask two questions of
the same noised data:

1. Can we recover what any **individual** user searched for? This should fail.
2. Can we recover what the **cohort as a whole** was shopping for? This should succeed.

In [18]:
import csv
from collections import Counter, defaultdict

# The 480 real user queries WANDS ships, grouped by their annotated class
with open(DATA_DIR / 'query.csv', newline='', encoding='utf-8') as handle:
    query_rows = list(csv.DictReader(handle, delimiter='\t'))

queries_by_class = defaultdict(list)
for row in query_rows:
    queries_by_class[row['query_class']].append(row['query'])

# Use the five best-populated classes as our user cohorts
COHORTS = [c for c, _ in Counter(
    {k: len(v) for k, v in queries_by_class.items()}).most_common(5)]

# Embed every query once, then project into the same PCA space as the index
cohort_queries = [q for c in COHORTS for q in queries_by_class[c]]
cohort_embedded = ((normalize(embedding_model.encode(cohort_queries)) - wands_pca['mean'])
                   @ wands_pca['components'].T)
query_vectors = dict(zip(cohort_queries, cohort_embedded))

print(f"{len(query_rows)} real queries, {len(queries_by_class)} classes. Using 5 as cohorts:\n")
for c in COHORTS:
    sample = ', '.join(queries_by_class[c][:3])
    print(f"  {c:<28} {len(queries_by_class[c]):>2} distinct queries   e.g. {sample}")

480 real queries, 189 classes. Using 5 as cohorts:

  Wall Art                     20 distinct queries   e.g. sunflower, 70s inspired furniture, wall art fiji
  Accent Chairs                16 distinct queries   e.g. leather chairs, tufted chair with gold legs, sancroft armchair
  Beds                         15 distinct queries   e.g. king poster bed, beds that have leds, full metal bed rose gold
  Area Rugs                    15 distinct queries   e.g. ombre rug, tollette teal outdoor rug, regner power loom red
  Coffee & Cocktail Tables     10 distinct queries   e.g. smart coffee table, westling coffee table, unique coffee tables


### 11a. One user hides, the crowd does not

Each simulated user picks one real query from their cohort and adds Laplace noise **on their own
device**. The server only ever sees noised vectors. We then compare two readouts of that same
data at `epsilon = 1.0`, which is heavy noise by the standards of section 10.

The cohort is identified by taking the mean of its noised vectors and matching it to the nearest
cohort centroid. The dominant-class readout is the more human version of the same question: of
the 25 products nearest that recovered centroid, which category do most of them belong to?

In [19]:
AGG_EPSILON = 1.0     # heavy noise: section 10 showed this leaks almost nothing per query
USERS_PER_COHORT = 20_000

rng = np.random.default_rng(0)

# Noiseless cohort centroids, used only to score the recovery
true_centroids = {c: np.mean([query_vectors[q] for q in queries_by_class[c]], axis=0)
                  for c in COHORTS}


def dominant_class(vector, k=25):
    """The category most common among the k products nearest this point."""
    distances = np.linalg.norm(wands_vectors - vector, axis=1)
    nearest = np.argpartition(distances, k)[:k]
    return Counter(wands_classes[nearest]).most_common(1)[0][0]


print(f"epsilon = {AGG_EPSILON}, {USERS_PER_COHORT:,} users per cohort\n")
header = f"{'cohort':<28}{'individual':>12}{'cohort ID':>12}   {'dominant class of recovered centroid'}"
print(header)
print('-' * len(header))

identified = 0
for cohort in COHORTS:
    picks = rng.choice(queries_by_class[cohort], size=USERS_PER_COHORT)
    raw = np.array([query_vectors[q] for q in picks])

    # Every user privatizes independently, client-side
    noised = raw + rng.laplace(0, 1.0 / AGG_EPSILON, size=raw.shape)

    # Readout 1: try to recover what a single user searched for
    individual_hits = [
        wands_classes[int(np.argmin(np.linalg.norm(wands_vectors - noised[i], axis=1)))] == cohort
        for i in range(300)
    ]

    # Readout 2: average the cohort, then match to the nearest cohort centroid
    centroid = noised.mean(axis=0)
    matched = min(COHORTS, key=lambda c: np.linalg.norm(centroid - true_centroids[c]))
    identified += matched == cohort

    print(f"{cohort:<28}{np.mean(individual_hits):>12.3f}"
          f"{('OK' if matched == cohort else 'MISS'):>12}   {dominant_class(centroid)}")

print(f"\nCohorts correctly identified from noised data: {identified}/{len(COHORTS)}")
print("Individual recovery stays near the random baseline of "
      f"{1 / len(np.unique(wands_classes)):.4f} throughout.")

epsilon = 1.0, 20,000 users per cohort

cohort                        individual   cohort ID   dominant class of recovered centroid
-------------------------------------------------------------------------------------------
Wall Art                           0.020          OK   Wall Art
Accent Chairs                      0.020          OK   Dining Chairs
Beds                               0.063          OK   Beds
Area Rugs                          0.070          OK   Area Rugs
Coffee & Cocktail Tables           0.030          OK   Coffee & Cocktail Tables

Cohorts correctly identified from noised data: 5/5
Individual recovery stays near the random baseline of 0.0012 throughout.


### 11b. Why it works: the noise averages away

The estimate improves as `1 / sqrt(n)`. That is not a property of this dataset, it is the
standard error of a mean, and it is what makes LDP practical at population scale. Doubling your
privacy budget is expensive. Collecting four times as many users costs nothing extra in privacy
and halves your error.

In [ ]:
sample_sizes = [10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000]
REPEATS = 5
CONV_COHORT = 'Beds'

rng = np.random.default_rng(1)
pool = queries_by_class[CONV_COHORT]

errors = []
for n in sample_sizes:
    trial_errors = []
    for _ in range(REPEATS):
        raw = np.array([query_vectors[q] for q in rng.choice(pool, size=n)])
        noised = raw + rng.laplace(0, 1.0 / AGG_EPSILON, size=raw.shape)
        trial_errors.append(np.linalg.norm(noised.mean(axis=0) - raw.mean(axis=0)))
    errors.append(np.mean(trial_errors))

# Theoretical 1/sqrt(n) curve, anchored at the first measured point
reference = [errors[0] * (sample_sizes[0] / n) ** 0.5 for n in sample_sizes]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.loglog(sample_sizes, errors, 'o-', color='tab:green',
          label='how far the noised average lands from the truth')
ax.loglog(sample_sizes, reference, '--', color='gray',
          label='what theory predicts, proportional to 1/sqrt(n)')
# The question this chart answers is "how many users do you need", so mark the
# answer on the x axis rather than drawing the accuracy bar on the y axis and
# making the room read the crossing sideways. The bar itself is unchanged: the
# mean per-coordinate spread of the index, a deliberately strict target.
accuracy_target = wands_vectors.std(axis=0).mean()
_x, _y = np.array(sample_sizes, float), np.array(errors)
_i = int(np.argmax(_y < accuracy_target))
smallest_segment = float(np.exp(np.interp(
    np.log(accuracy_target), np.log(_y[[_i, _i - 1]]), np.log(_x[[_i, _i - 1]]))))
ax.axvline(smallest_segment, color='tab:red', linestyle='dashed', linewidth=2,
           label=f'{round(smallest_segment, -2):,.0f} users, the smallest measurable segment')
# A log axis labels itself 10^1 ... 10^5, which leaves the sampled 30, 300, 3k
# and 30k between unlabelled decades. Label the sizes actually measured.
ax.set_xticks(sample_sizes)
ax.set_xticklabels([f'{n}' if n < 1000 else f'{n // 1000:g}k' for n in sample_sizes])
ax.tick_params(axis='x', which='minor', bottom=False, labelbottom=False)
ax.set_xlabel('Users whose noised queries are averaged together')
ax.set_ylabel("Distance from the segment's true center")
ax.set_title(f"Averaging more users recovers the truth (epsilon = {AGG_EPSILON}, segment '{CONV_COHORT}')")
ax.legend()
ax.grid(alpha=0.3, which='both')
fig.savefig('plots/11_aggregate_convergence.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"{'users':>8}{'centroid error':>17}")
for n, e in zip(sample_sizes, errors):
    print(f"{n:>8,}{e:>17.4f}")


### What this means for relevance work

The red line marks the spread of the index itself. Once the estimation error drops well below
it, the recovered centroid points at a specific neighbourhood of the catalogue rather than a
vague direction, and the aggregate becomes usable.

So the constraint LDP imposes is not "you lose your analytics." It is **you lose the ability to
ask about one person**. Anything you can phrase as a population statistic still works, and those
are exactly the inputs relevance tuning depends on.

Two honest limits are worth stating alongside the win. The dominant-class readout puts the
Accent Chairs cohort in an adjacent chair category, which is the same category-level blurring
section 10 measured, and it does not sharpen with more users because it is bias rather than
noise. And low-traffic segments do not reach the population sizes this depends on, so the tail
of your query distribution stays genuinely unmeasurable.